In [0]:
use tpc_a_dev.gold

In [0]:
select * from tpc_a_dev.gold.fact_bookings

In [0]:
-- Overall KPI summary
Select 
    Sum(ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100)) as total_net_revenue,
    count(Distinct booking_id) as distinct_bookings,
    count(Distinct booking_id) as active_passengers,
    avg(ticket_price * seats) as avg_booking_value
From tpc_a_dev.gold.fact_bookings
where refund_flag = 'N'

In [0]:
-- Monthly revenue trend across all years
Select 
    Year(commit_date) as year,
    month (commit_date) as month,
    sum (ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100)) as monthly_revenue
From tpc_a_dev.gold.fact_bookings
where refund_flag = 'N'
Group By year, month
order By year, month

In [0]:
-- Annual revenue with year-over-year change
Select 
    Year(commit_date) as year,
    sum(ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100)) as annual_revenue
From tpc_a_dev.gold.fact_bookings
where refund_flag = 'N'
group by year
order by year


In [0]:
%sql
SELECT 
    p.travel_purpose,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p
  ON f.booking_id = p.passenger_id
WHERE f.refund_flag = 'N'
GROUP BY p.travel_purpose
ORDER BY revenue DESC;

In [0]:
SELECT 
    p.passenger_id,
    p.name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS passenger_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p
  ON f.booking_id = p.passenger_id
WHERE f.refund_flag = 'N'
GROUP BY p.passenger_id, p.name
ORDER BY passenger_revenue DESC
LIMIT 10;

In [0]:
WITH top_passengers AS (
  SELECT p.passenger_id, SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS passenger_revenue
  FROM tpc_a_dev.gold.fact_bookings f
  JOIN tpc_a_dev.gold.dim_passenger p
    ON f.booking_id = p.passenger_id
  WHERE f.refund_flag = 'N'
  GROUP BY p.passenger_id
  ORDER BY passenger_revenue DESC
  LIMIT 10
)
SELECT 
  SUM(passenger_revenue) / (
    SELECT SUM(ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100)) 
    FROM tpc_a_dev.gold.fact_bookings
    WHERE refund_flag = 'N'
  ) * 100 AS top_10_share_pct
FROM top_passengers;

In [0]:
SELECT 
    r.route_id,
    r.name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS route_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_route r
  ON f.route_id = r.route_id
WHERE f.refund_flag = 'N'
GROUP BY r.route_id, r.name
ORDER BY route_revenue DESC
LIMIT 10;

In [0]:
SELECT 
    r.route_id,
    r.name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS route_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_route r
  ON f.route_id = r.route_id
WHERE f.refund_flag = 'N'
GROUP BY r.route_id, r.name
ORDER BY route_revenue ASC
LIMIT 10;

In [0]:
SELECT 
    r.route_type,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS route_type_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_route r ON f.route_id = r.route_id
WHERE f.refund_flag = 'N'
GROUP BY r.route_type
ORDER BY route_type_revenue DESC;

In [0]:
SELECT 
    a.airline_id,
    a.name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS airline_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_airline a ON f.airline_id = a.airline_id
WHERE f.refund_flag = 'N'
GROUP BY a.airline_id, a.name
ORDER BY airline_revenue DESC
LIMIT 10;

In [0]:
WITH top_airlines AS (
  SELECT a.airline_id, SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS airline_revenue
  FROM tpc_a_dev.gold.fact_bookings f
  JOIN tpc_a_dev.gold.dim_airline a ON f.airline_id = a.airline_id
  WHERE f.refund_flag = 'N'
  GROUP BY a.airline_id
  ORDER BY airline_revenue DESC
  LIMIT 10
)
SELECT 
  SUM(airline_revenue) / (
    SELECT SUM(ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100))
    FROM tpc_a_dev.gold.fact_bookings
    WHERE refund_flag = 'N'
  ) * 100 AS top_10_airlines_share_pct
FROM top_airlines;

In [0]:
SELECT 
    g.region_name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS region_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p ON f.booking_id = p.passenger_id
JOIN tpc_a_dev.gold.dim_geography g ON p.nation_key = g.nation_id
WHERE f.refund_flag = 'N'
GROUP BY g.region_name
ORDER BY region_revenue DESC;

In [0]:
SELECT 
    g.nation_name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS nation_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p ON f.booking_id = p.passenger_id
JOIN tpc_a_dev.gold.dim_geography g ON p.nation_key = g.nation_id
WHERE f.refund_flag = 'N'
GROUP BY g.nation_name
ORDER BY nation_revenue DESC;

In [0]:
SELECT 
    refund_flag,
    SUM(ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100)) AS revenue,
    COUNT(*) AS total_bookings
FROM tpc_a_dev.gold.fact_bookings
GROUP BY refund_flag
ORDER BY revenue DESC;

In [0]:
SELECT 
    p.travel_purpose,
    AVG(f.discount_pct) AS avg_discount
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p ON f.booking_id = p.passenger_id
WHERE f.refund_flag = 'N'
GROUP BY p.travel_purpose
ORDER BY avg_discount DESC;

In [0]:
SELECT 
    cabin_class,
    AVG(DATEDIFF(arrival_date, departure_date)) AS avg_duration_days
FROM tpc_a_dev.gold.fact_bookings
WHERE refund_flag = 'N'
GROUP BY cabin_class
ORDER BY avg_duration_days DESC;

In [0]:
SELECT 
    segment_status AS booking_priority,
    COUNT(*) AS booking_count,
    SUM(ticket_price * seats * (1 - discount_pct/100) * (1 + tax_pct/100)) AS total_revenue
FROM tpc_a_dev.gold.fact_bookings
WHERE refund_flag = 'N'
GROUP BY segment_status
ORDER BY total_revenue DESC;

In [0]:
SELECT 
    YEAR(commit_date) AS year,
    MONTH(commit_date) AS month,
    segment_status,
    COUNT(*) AS booking_count
FROM tpc_a_dev.gold.fact_bookings
GROUP BY year, month, segment_status
ORDER BY year, month, segment_status;

In [0]:
SELECT 
    p.travel_purpose,
    r.route_id,
    r.name,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS route_revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p ON f.booking_id = p.passenger_id
JOIN tpc_a_dev.gold.dim_route r ON f.route_id = r.route_id
WHERE f.refund_flag = 'N'
GROUP BY p.travel_purpose, r.route_id, r.name
ORDER BY p.travel_purpose, route_revenue DESC;

In [0]:
SELECT 
    g.region_name AS passenger_region,
    ag.region_name AS airline_region,
    SUM(f.ticket_price * f.seats * (1 - f.discount_pct/100) * (1 + f.tax_pct/100)) AS revenue
FROM tpc_a_dev.gold.fact_bookings f
JOIN tpc_a_dev.gold.dim_passenger p ON f.booking_id = p.passenger_id
JOIN tpc_a_dev.gold.dim_geography g ON p.nation_key = g.nation_id
JOIN tpc_a_dev.gold.dim_airline a ON f.airline_id = a.airline_id
JOIN tpc_a_dev.gold.dim_geography ag ON a.nation_key = ag.nation_id
WHERE f.refund_flag = 'N'
GROUP BY g.region_name, ag.region_name
ORDER BY revenue DESC;